# PQ-EVM vs EVM — Benchmark & Cost Analysis

A side-by-side comparison of a post-quantum EVM (PQ-EVM) against the classical EVM, focusing on the practical costs of migrating Ethereum-style cryptography from elliptic curves to NIST-standardized lattice schemes.

**Schemes under test**

| Component | Classical | Post-Quantum |
|---|---|---|
| Signature | ECDSA / secp256k1 | ML-DSA-65 (FIPS 204, NIST Level 3) |
| Hash      | Keccak-256        | SHAKE-256 (FIPS 202)               |
| Consensus | PoS (~12.8 min finality) | PoA (~5 s finality)          |

**Methodology.** Cryptographic micro-benchmarks were measured with [Criterion.rs](https://bheisler.github.io/criterion.rs/book/) in release mode (LLVM `-O3`, `target-cpu=native`, AVX2 enabled). Each data point is the mean of 100 samples after warm-up. Gas costs were calibrated empirically against `ecrecover` on a modified Reth client.

**Headline finding.** The computational overhead of post-quantum cryptography is *not* the bottleneck. ML-DSA verification is even **faster** than `ecrecover`. The real cost is **spatial** — transactions grow ~87× — which dominates bandwidth, storage and per-block throughput.


In [ ]:
# Run this notebook with the qiskit-api venv to avoid reinstalling deps:
#   uv run --project ../qiskit-api jupyter lab docs/benchmark_analysis.ipynb

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# --- Style (no seaborn dependency) ---
plt.style.use("default")
plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "semibold",
    "axes.labelsize": 11,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.alpha": 0.25,
    "grid.linestyle": "--",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

# --- Color palette ---
C_CLASSICAL = "#1976D2"   # Blue   — classical / ECDSA / Keccak
C_PQ        = "#E64A19"   # Orange — post-quantum / ML-DSA / SHAKE
C_GOOD      = "#2E7D32"   # Green  — PQ wins
C_NEUTRAL   = "#757575"   # Gray   — reference
C_ACCENT    = "#7B1FA2"   # Purple — annotations / projections

FIG_DIR = "."  # relative to docs/

def save(fig, name):
    fig.savefig(f"{FIG_DIR}/{name}.png", dpi=200, bbox_inches="tight")


## 1. The headline result — ML-DSA-65 verification is faster than `ecrecover`

Verification is the operation that matters most: every full node executes it for every transaction. Counter-intuitively, lattice-based verification beats elliptic-curve key recovery by ~14 % on modern x86 (AVX2). ML-DSA's hot path is dominated by NTT-friendly polynomial arithmetic that vectorizes extremely well; secp256k1 recovery requires a modular inverse plus a scalar multiplication that is hard to parallelize.

In [ ]:
operations = ["Key generation", "Signing", "Verification"]
classical_us = [35.1, 41.4, 49.2]   # microseconds, ECDSA secp256k1
pq_us        = [208.5, 342.8, 42.0] # microseconds, ML-DSA-65

fig, axes = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [1.3, 1]})

# --- Left: full lifecycle comparison ---
ax = axes[0]
x = np.arange(len(operations))
w = 0.36
b1 = ax.bar(x - w/2, classical_us, w, label="ECDSA / secp256k1",
            color=C_CLASSICAL, edgecolor="white", linewidth=0.8)
b2 = ax.bar(x + w/2, pq_us, w, label="ML-DSA-65",
            color=C_PQ, edgecolor="white", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(operations)
ax.set_ylabel("Time (\u00b5s, lower is better)")
ax.set_title("Full lifecycle: classical vs post-quantum")
ax.bar_label(b1, fmt="%.1f", padding=2, fontsize=9)
ax.bar_label(b2, fmt="%.1f", padding=2, fontsize=9)
ax.legend(loc="upper left")

# Annotate the relative ratios above each pair
for i, (c, p) in enumerate(zip(classical_us, pq_us)):
    ratio = p / c
    sym = "\u2715"  # ×
    color = C_GOOD if ratio < 1 else C_PQ
    ax.text(i, max(c, p) * 1.08, f"{ratio:.2f}{sym}", ha="center",
            color=color, fontsize=10, fontweight="bold")
ax.set_ylim(0, max(pq_us) * 1.25)

# --- Right: zoom on verification (the critical operation) ---
ax = axes[1]
verify = {
    "ecrecover\n(secp256k1)": 49.2,
    "ECDSA verify\n(secp256k1)": 45.8,
    "ML-DSA-65\nverify": 42.0,
}
colors = [C_CLASSICAL, C_CLASSICAL, C_PQ]
bars = ax.bar(verify.keys(), verify.values(), color=colors,
              edgecolor="black", linewidth=0.4)
ax.set_ylabel("Time (\u00b5s)")
ax.set_title("Verification — the hot path for every node")
ax.set_ylim(0, 60)
ax.bar_label(bars, fmt="%.1f \u00b5s", padding=3, fontsize=10, fontweight="bold")
ax.annotate("14% faster", xy=(2, 42), xytext=(2, 56),
            ha="center", fontsize=11, fontweight="bold", color=C_PQ,
            arrowprops=dict(arrowstyle="->", color=C_PQ, lw=2))

fig.suptitle("Cryptographic operation latency on x86_64 + AVX2", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save(fig, "fig1_verification_speed")
plt.show()


## 2. Hashing — SHAKE-256 converges with Keccak-256 around 4 KB

SHAKE-256 is ~1.8× slower for tiny inputs (32 B), but the gap closes as the message grows: at 4 KB the two are within 0.5 %, and at 8 KB SHAKE is marginally faster. Since post-quantum public keys are 1 952 B, the practical overhead is only ~13 %.

In [ ]:
sizes      = [32, 64, 128, 256, 512, 1024, 4096, 8192]
shake_ns   = [545, 560, 573, 802, 1322, 2329, 8100, 15608]
keccak_ns  = [300, 304, 301, 555, 1089, 2054, 8130, 16229]
labels     = ["32 B", "64 B", "128 B", "256 B", "512 B", "1 KB", "4 KB", "8 KB"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left: log-log absolute timings ---
ax = axes[0]
ax.plot(sizes, shake_ns, "o-", color=C_PQ, lw=2, ms=7, label="SHAKE-256")
ax.plot(sizes, keccak_ns, "s-", color=C_CLASSICAL, lw=2, ms=7, label="Keccak-256")
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xlabel("Input size (bytes, log\u2082)")
ax.set_ylabel("Time (ns, log)")
ax.set_title("Absolute hashing latency")
ax.set_xticks(sizes)
ax.set_xticklabels(labels, rotation=0)
ax.axvspan(4096, 8192, alpha=0.10, color=C_GOOD)
ax.text(5500, 350, "convergence\nzone", ha="center", fontsize=9, color=C_GOOD)
ax.legend(loc="upper left")

# --- Right: relative ratio SHAKE / Keccak ---
ax = axes[1]
ratios = [s / k for s, k in zip(shake_ns, keccak_ns)]
colors = [C_PQ if r > 1.0 else C_GOOD for r in ratios]
bars = ax.bar(labels, ratios, color=colors, edgecolor="black", linewidth=0.4)
ax.axhline(1.0, color="black", linestyle="--", lw=1, label="Parity")
ax.set_xlabel("Input size")
ax.set_ylabel("Ratio (SHAKE-256 / Keccak-256)")
ax.set_title("Relative overhead of SHAKE-256")
ax.bar_label(bars, fmt="%.2fx", padding=2, fontsize=9)
ax.set_ylim(0, 2.2)
ax.legend(loc="upper right")
ax.annotate("PQ public key (1 952 B)\nratio \u2248 1.13x",
            xy=(5, 1.13), xytext=(5.5, 1.75),
            ha="center", fontsize=9, color=C_GOOD,
            arrowprops=dict(arrowstyle="->", color=C_GOOD))

fig.suptitle("SHAKE-256 vs Keccak-256 — message-size sensitivity", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save(fig, "fig2_hash_comparison")
plt.show()


## 3. The real cost — transactions are 87× larger

Computational overhead is negligible. The genuine bottleneck is **size**: a single PQ transfer is 5.3 KB versus 61 B for the classical one. This dominates bandwidth, mempool memory, state-diff propagation and long-term storage.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6), gridspec_kw={"width_ratios": [1, 1.2]})

# --- Left: stacked transaction anatomy ---
ax = axes[0]
classical_parts = {"Nonce + gas + value": 30, "To address": 20, "Signature (v,r,s)": 65}
pq_parts        = {"Nonce + gas + value + chainID": 53, "To address": 20,
                   "Signature": 3309, "Public key": 1952}
categories = ["Classical\n(ECDSA)", "Post-quantum\n(ML-DSA-65)"]

classical_palette = ["#90CAF9", "#42A5F5", "#1565C0"]
pq_palette        = ["#FFAB91", "#FF7043", "#E64A19", "#BF360C"]

for col, parts, palette in [(0, classical_parts, classical_palette),
                            (1, pq_parts, pq_palette)]:
    bottom = 0
    for i, (label, val) in enumerate(parts.items()):
        ax.bar(col, val, bottom=bottom, color=palette[i], edgecolor="white",
               linewidth=0.8, width=0.55)
        if val > 80:  # only annotate large segments inside the bar
            ax.text(col, bottom + val/2, f"{label}\n{val} B",
                    ha="center", va="center", fontsize=9,
                    color="white" if val > 200 else "black")
        bottom += val

ax.set_xticks([0, 1])
ax.set_xticklabels(categories)
ax.set_ylabel("Transaction size (bytes)")
ax.set_title("Transaction anatomy — what eats the space")
ax.text(0.5, 4900, "87\u00d7 larger", ha="center", fontsize=14,
        fontweight="bold", color=C_PQ)
ax.annotate("", xy=(1, 5314), xytext=(0, 115),
            arrowprops=dict(arrowstyle="->", color=C_PQ, lw=1.5,
                            connectionstyle="arc3,rad=-0.25"))

# --- Right: resource impact at the network level ---
ax = axes[1]
metrics = ["Tx size (B)", "Gas cost\n(transfer)", "Block capacity\n(KB)",
           "Bandwidth\n(per node)", "Daily\nstorage", "TPS\n(max)"]
classical_vals = [61, 21976, 100, 1.0, 2.9, 546]   # last unit varies; we plot ratios
pq_vals        = [5314, 42892, 4500, 45.0, 130, 280]
ratios_block   = [pq / c for pq, c in zip(pq_vals, classical_vals)]

y = np.arange(len(metrics))
bar_colors = [C_PQ if r > 1 else C_GOOD for r in ratios_block]
bars = ax.barh(y, ratios_block, color=bar_colors, edgecolor="black", linewidth=0.3)
ax.axvline(1.0, color="black", linestyle="--", lw=1)
ax.set_yticks(y)
ax.set_yticklabels(metrics)
ax.set_xlabel("Ratio (PQ / classical)  \u2014 logarithmic")
ax.set_xscale("log")
ax.set_title("Network-level resource impact")
for r, bar in zip(ratios_block, bars):
    ax.text(r * 1.1, bar.get_y() + bar.get_height()/2, f"{r:.1f}\u00d7",
            ha="left", va="center", fontsize=10, fontweight="bold")
ax.set_xlim(0.3, max(ratios_block) * 4)
ax.invert_yaxis()  # most impactful at top

fig.suptitle("Spatial overhead — the dominant cost of post-quantum migration",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save(fig, "fig3_tx_size_impact")
plt.show()


## 4. Gas calibration — from 50 000 → 3 450 gas

The ML-DSA-65 verification precompile (address `0x0100`) was originally a 50 000-gas placeholder. Empirical timing against existing precompiles shows a fair price closer to **3 450 gas** — only marginally above `ecrecover` (3 000), and well below the original guess. Gas is meant to track CPU cost, and CPU cost is essentially equivalent.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

precompiles = ["ecrecover\n(0x01)", "SHA-256\n(0x02)", "RIPEMD-160\n(0x03)",
               "Identity\n(0x04)", "ModExp\n(0x05)", "ML-DSA verify\n(0x0100)"]
gas_costs = [3000, 60, 600, 15, 200, 3450]
colors_gas = [C_CLASSICAL, C_NEUTRAL, C_NEUTRAL, C_NEUTRAL, C_NEUTRAL, C_PQ]

bars = ax.bar(precompiles, gas_costs, color=colors_gas, edgecolor="black", linewidth=0.4)
ax.set_ylabel("Gas")
ax.set_title("Precompile gas costs — classical EVM vs PQ-EVM")
ax.bar_label(bars, fmt="%d", padding=3, fontsize=10, fontweight="bold")

# --- Annotation: the calibration delta ---
ax.annotate("", xy=(5, 3450), xytext=(5, 50000),
            arrowprops=dict(arrowstyle="->", color="red", lw=2))
ax.text(5.35, 26000, "Calibrated\n50 000 \u2192 3 450\n(\u221293%)",
        fontsize=10, color="red", ha="left", va="center", fontweight="bold")
ax.set_ylim(0, 56000)
ax.axhline(3000, color=C_CLASSICAL, linestyle=":", lw=1, alpha=0.7)
ax.text(0.05, 3300, "ecrecover baseline (3 000)", fontsize=8,
        color=C_CLASSICAL, alpha=0.85, transform=ax.get_yaxis_transform())

plt.tight_layout()
save(fig, "fig4_gas_calibration")
plt.show()


## 5. Security level — what we get for the extra bytes

Cost without context is meaningless. Here is what each scheme actually buys against a classical *and* a quantum adversary. ECDSA's apparent strength (~128-bit) collapses to **0** under Shor's algorithm — a sufficiently large quantum computer recovers the private key in polynomial time. ML-DSA-65 is the NIST Level-3 lattice scheme: ~192-bit security against both classical and quantum attack.

In [ ]:
schemes = ["ECDSA\nsecp256k1", "RSA-2048", "ML-DSA-44\n(NIST L2)",
           "ML-DSA-65\n(NIST L3)", "ML-DSA-87\n(NIST L5)"]
classical_bits = [128, 112, 128, 192, 256]
quantum_bits   = [0,   0,   128, 192, 256]   # Shor breaks ECDSA & RSA
is_pq          = [False, False, True, True, True]

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), gridspec_kw={"width_ratios": [1.4, 1]})

# --- Left: classical vs quantum security per scheme ---
ax = axes[0]
x = np.arange(len(schemes))
w = 0.36
b1 = ax.bar(x - w/2, classical_bits, w, label="Classical security",
            color=C_NEUTRAL, edgecolor="white", linewidth=0.8)
b2 = ax.bar(x + w/2, quantum_bits, w, label="Quantum security (post-Shor)",
            color=C_ACCENT, edgecolor="white", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(schemes)
ax.set_ylabel("Security (bits)")
ax.set_title("Effective security: classical adversary vs quantum adversary")
ax.bar_label(b1, fmt="%d", padding=2, fontsize=9)
ax.bar_label(b2, fmt="%d", padding=2, fontsize=9)
ax.axhline(80,  color="orange", linestyle=":", lw=1, alpha=0.6)
ax.text(-0.5, 82, "deprecated (80-bit)", fontsize=8, color="orange")
ax.axhline(128, color=C_GOOD, linestyle=":", lw=1, alpha=0.6)
ax.text(-0.5, 130, "recommended floor (128-bit)", fontsize=8, color=C_GOOD)
ax.legend(loc="upper left")
ax.set_ylim(0, 290)

# Mark broken schemes
for i, broken in enumerate([True, True, False, False, False]):
    if broken:
        ax.text(i + w/2, 6, "BROKEN", ha="center", color="red",
                fontsize=8, fontweight="bold")

# --- Right: cost vs security trade-off (security per byte of signature) ---
ax = axes[1]
sig_bytes = [65, 256, 2420, 3309, 4627]
sec_per_byte = [q / s for q, s in zip(quantum_bits, sig_bytes)]
colors = [C_CLASSICAL if not pq else C_PQ for pq in is_pq]
ax.scatter(sig_bytes, quantum_bits, s=180, c=colors, edgecolor="black", linewidth=0.6, zorder=3)
for sx, sy, label in zip(sig_bytes, quantum_bits, schemes):
    ax.annotate(label.replace("\n", " "), (sx, sy),
                xytext=(8, 8), textcoords="offset points", fontsize=9)
ax.set_xlabel("Signature size (bytes)")
ax.set_ylabel("Quantum security (bits)")
ax.set_title("Signature size vs post-quantum security")
ax.set_xscale("log")
ax.set_xlim(40, 8000)
ax.set_ylim(-20, 290)
ax.axhspan(-20, 0, alpha=0.08, color="red")
ax.text(80, -10, "broken by Shor", fontsize=9, color="red")

fig.suptitle("Security gained per byte spent — the value side of the trade-off",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save(fig, "fig5_security_levels")
plt.show()


## 6. Network-scale projection — what migration would cost Ethereum mainnet

Ethereum mainnet processes roughly **1.2 M transactions/day** at ~30 M gas/block and a 12-second block time. The chart below extrapolates the per-tx PQ overhead into the metrics that operators actually feel: bandwidth, monthly chain growth, and the realistic TPS ceiling under a fixed gas limit.

In [ ]:
# --- Mainnet baseline assumptions ---
BLOCK_TIME_S      = 12
GAS_LIMIT         = 30_000_000
DAILY_TX_CLASSICAL = 1_200_000  # current mainnet load
BYTES_CLASSICAL    = 61
BYTES_PQ           = 5314
GAS_CLASSICAL_TX   = 21_000  # plain transfer
GAS_PQ_TX          = 21_000 + 3_450 + 18_500  # tx base + ML-DSA verify + extra calldata

# --- Derived metrics ---
max_tps_classical = GAS_LIMIT / GAS_CLASSICAL_TX / BLOCK_TIME_S
max_tps_pq        = GAS_LIMIT / GAS_PQ_TX / BLOCK_TIME_S
daily_tx_pq       = DAILY_TX_CLASSICAL * (max_tps_pq / max_tps_classical)

daily_bytes_classical = DAILY_TX_CLASSICAL * BYTES_CLASSICAL
daily_bytes_pq        = DAILY_TX_CLASSICAL * BYTES_PQ  # same load, PQ-encoded
monthly_growth_classical = daily_bytes_classical * 30 / 1e9   # GB
monthly_growth_pq        = daily_bytes_pq * 30 / 1e9
yearly_growth_classical  = daily_bytes_classical * 365 / 1e12 # TB
yearly_growth_pq         = daily_bytes_pq * 365 / 1e12

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# --- Left: TPS ceiling under fixed gas limit ---
ax = axes[0]
categories = ["Classical EVM", "PQ-EVM"]
values = [max_tps_classical, max_tps_pq]
colors = [C_CLASSICAL, C_PQ]
bars = ax.bar(categories, values, color=colors, edgecolor="black", linewidth=0.4, width=0.55)
ax.set_ylabel("Theoretical max TPS (transfers)")
ax.set_title(f"Throughput ceiling at {GAS_LIMIT/1e6:.0f}M gas/block, {BLOCK_TIME_S}s block time")
ax.bar_label(bars, fmt="%.0f tps", padding=3, fontsize=11, fontweight="bold")
ax.set_ylim(0, max(values) * 1.25)
ax.text(1, max_tps_pq * 1.5,
        f"{(1 - max_tps_pq/max_tps_classical)*100:.0f}% throughput loss",
        ha="center", color=C_PQ, fontweight="bold", fontsize=11)

# --- Right: chain growth projection (5 years) ---
ax = axes[1]
years = np.arange(0, 6)
growth_classical = years * yearly_growth_classical
growth_pq        = years * yearly_growth_pq
ax.fill_between(years, growth_classical, color=C_CLASSICAL, alpha=0.25,
                label=f"Classical (~{yearly_growth_classical:.2f} TB/yr)")
ax.fill_between(years, growth_pq, color=C_PQ, alpha=0.25,
                label=f"PQ-EVM (~{yearly_growth_pq:.1f} TB/yr)")
ax.plot(years, growth_classical, color=C_CLASSICAL, lw=2, marker="o")
ax.plot(years, growth_pq, color=C_PQ, lw=2, marker="o")
ax.set_xlabel("Years from migration")
ax.set_ylabel("Cumulative tx data (TB)")
ax.set_title("5-year chain growth at current mainnet load (1.2 M tx/day)")
ax.legend(loc="upper left")
ax.set_xticks(years)
ax.annotate(f"{growth_pq[-1]:.1f} TB after 5 yr",
            xy=(5, growth_pq[-1]), xytext=(3.0, growth_pq[-1] * 0.6),
            fontsize=10, color=C_PQ, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=C_PQ))

fig.suptitle("Mainnet projection — what operators actually pay",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save(fig, "fig6_mainnet_projection")
plt.show()

print(f"Classical max TPS:  {max_tps_classical:7.1f}")
print(f"PQ-EVM max TPS:     {max_tps_pq:7.1f}")
print(f"Throughput loss:    {(1 - max_tps_pq/max_tps_classical)*100:5.1f}%")
print(f"Chain growth/year:  {yearly_growth_classical*1e3:6.1f} GB (classical) -> {yearly_growth_pq*1e3:7.1f} GB (PQ)")


## 7. Trade-off summary

The post-quantum migration is a clean *time-vs-space* trade. Computationally we lose almost nothing — and verification actually gains. We pay almost everything in bytes.

In [ ]:
categories = [
    "Verification speed",
    "Hash cost (\u22651 KB)",
    "Gas cost (precompile)",
    "Finality (PoA vs PoS)",
    "Quantum security",
    "Hash cost (32 B)",
    "Key generation",
    "Signing speed",
    "Tx size",
    "Block capacity (TPS)",
    "Bandwidth per node",
    "Storage growth",
]
# Signed score: positive = PQ better, negative = PQ worse (clipped to ±100 visual scale)
improvements = [14, 4, -15, 60, 100, -45, -83, -88, -98, -49, -97, -97]
colors = [C_GOOD if v >= 0 else C_PQ for v in improvements]

fig, ax = plt.subplots(figsize=(11, 6.5))
y = np.arange(len(categories))
bars = ax.barh(y, improvements, color=colors, edgecolor="black", linewidth=0.3, alpha=0.88)
ax.axvline(0, color="black", lw=1.5)
ax.set_yticks(y)
ax.set_yticklabels(categories)
ax.set_xlabel("\u2190  PQ worse                                                PQ better  \u2192")
ax.set_title("PQ-EVM vs classical EVM — trade-off summary", fontweight="bold")
ax.invert_yaxis()  # winners at top

# Annotate
ax.text(60, 0.7, "Time / security:\nessentially free",
        fontsize=10, style="italic", color=C_GOOD)
ax.text(-60, 11.4, "Space:\nthe real cost",
        fontsize=10, style="italic", color=C_PQ, ha="center")

legend_handles = [
    mpatches.Patch(color=C_GOOD, label="PQ advantage"),
    mpatches.Patch(color=C_PQ, label="PQ disadvantage (spatial)"),
]
ax.legend(handles=legend_handles, loc="lower right")

plt.tight_layout()
save(fig, "fig7_tradeoff_summary")
plt.show()


## Key takeaways

1. **ML-DSA-65 verification is faster than `ecrecover`** (42 µs vs 49 µs, −14 %). Lattice arithmetic vectorizes well on AVX2; elliptic-curve key recovery does not.
2. **SHAKE-256 converges with Keccak-256** for inputs ≥ 4 KB. At the 1 952-byte public-key size the overhead is only ~13 %.
3. **The real cost is spatial, not computational.** Transactions are 87× larger, which propagates into ~45× bandwidth, ~45× storage, and a ~50 % hit to throughput at a fixed gas limit.
4. **Gas pricing is fair.** ML-DSA verification at 3 450 gas reflects measured CPU cost; the original 50 000-gas placeholder over-charged by ~14×.
5. **Security gain is asymmetric.** Classical schemes drop to 0 bits under Shor's algorithm. ML-DSA-65 holds 192 bits against both classical and quantum adversaries.
6. **Migration is operationally viable.** At current mainnet load the chain grows ~2.3 TB/year of tx data instead of ~27 GB/year — an ~85× jump that is still absorbable on commodity NVMe today, and shrinks dramatically once paired with rollups, signature aggregation, or stateless clients.

## Next steps

- Re-run the throughput projection with **EIP-4844 blobs** to estimate effective TPS once PQ signatures live in the blob region instead of calldata.
- Benchmark **batched ML-DSA verification** — amortizing NTT setup across N signatures should move the needle further on the verify side.
- Profile the **Reth state-trie write path** under a steady stream of PQ-sized transactions; storage IO is likely the next bottleneck after bandwidth.
